In [7]:
import geopandas as gpd
import rasterio
import shapely

print("geopandas", gpd.__version__)
print("rasterio ", rasterio.__version__, "| GDAL", rasterio.__gdal_version__)
print("shapely  ", shapely.__version__)

geopandas 1.2.0
rasterio  1.5.2 | GDAL 3.12.2
shapely   2.2.0


A1. Geometria i GeoDataFrame od zera

In [8]:
from shapely.geometry import Point, Polygon

rynek = Point(17.9226, 50.6681)          # (x, y) = (longitude, latitude)
square = Polygon([(17.92, 50.66), (17.93, 50.66), (17.93, 50.67), (17.92, 50.67)])

print(rynek)
print(square.area)                       # in "units of the coordinates" - here: square degrees!
print(square.contains(rynek))

POINT (17.9226 50.6681)
0.00010000000000003127
True


In [9]:
import geopandas as gpd

places = gpd.GeoDataFrame(
    {"name": ["Rynek", "Square"]},
    geometry=[rynek, square],
    crs="EPSG:4326",                     # WGS 84 - longitude/latitude in degrees
)
places
print(type(places))
print(places.crs)
print(places.geom_type)

<class 'geopandas.geodataframe.GeoDataFrame'>
EPSG:4326
0      Point
1    Polygon
dtype: str


A2. Wczytanie surowych danych

In [17]:
import pandas as pd

raw = pd.read_csv("dane/pm25_sensors.csv", parse_dates=["measured_at"])
print(raw.shape)
raw.head(10)

#Pierwsza zasada pracy z danymi z zewnątrz: zanim cokolwiek policzysz, obejrzyj dane.

raw[["lat", "lon", "pm25"]].describe()
raw.isna().sum()
raw.duplicated().sum()

(145, 5)


np.int64(5)

A3. Czyszczenie danych

In [18]:
df = raw.copy()
report = {"input rows": len(df)}

# 1. Duplicates - the same message sent twice
df = df.drop_duplicates()
report["after drop_duplicates"] = len(df)

# 2. Rows without coordinates cannot be placed on a map
df = df.dropna(subset=["lat", "lon"])
report["after dropna"] = len(df)

# 3. Swapped coordinates: in Poland lat is 49-55 and lon is 14-24
swapped = df["lat"].between(14, 24.5) & df["lon"].between(49, 55)
df.loc[swapped, ["lat", "lon"]] = df.loc[swapped, ["lon", "lat"]].to_numpy()
report["swapped rows fixed"] = int(swapped.sum())

# 4. -999 is the sensor's error code, not a measurement
df = df[df["pm25"] >= 0].reset_index(drop=True)
report["clean rows"] = len(df)

report
# Zwróć uwagę na .to_numpy() w punkcie 3. Bez niego pandas dopasowałby kolumny po nazwach i zamiana by się „odwróciła” - spróbuj usunąć .to_numpy() i zobacz, co się stanie.

df[["lat", "lon", "pm25"]].describe().round(2)

,lat,lon,pm25
count,130.00,130.00,130.00
mean,50.67,17.92,18.86
std,0.03,0.04,5.19
min,50.62,17.79,9.20
25%,50.65,17.90,14.65
50%,50.67,17.92,19.00
75%,50.69,17.94,22.22
max,50.77,18.02,35.30


A4. Z tabeli do GeoDataFrame

In [19]:
sensors = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),   # x = lon, y = lat!
    crs="EPSG:4326",
)
sensors.head()
print(sensors.geom_type.unique())
print(sensors.crs)
print(sensors.total_bounds)              # [minx, miny, maxx, maxy]

# Szybki podgląd na mapie (w Jupyterze wyświetli się interaktywna mapa):

sensors.explore(column="pm25", cmap="Oranges", marker_kwds={"radius": 6})

<ArrowStringArray>
['Point']
Length: 1, dtype: str
EPSG:4326
[17.785622 50.616593 18.020378 50.767895]
